# Day 10 — HOL 2: Incremental Gold Refresh — MERGE-Based Fact Table Updates

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Follows** | HOL 1 (SCD2 MERGE for `dim_customer`), Day 7 (the full-load `fact_sales` build) |
| **Source** | `gbmart.silver.order_items/orders/products/address/payments`, `gbmart.gold.dim_date` (read-only — cloned below) |
| **Duration** | ~90 minutes |
| **Output** | Your own scratch-schema `fact_sales` table, refreshed incrementally instead of rebuilt from scratch |

### Learning Objectives
- Apply the same `MERGE` technique from HOL 1 to a **fact** table, not just a dimension
- Identify only the changed `order_items` rows via CDF, instead of reprocessing all ~377K rows
- Understand when an incremental MERGE refresh is worth the extra complexity over a full overwrite

---
**Recap — how Day 7 built `fact_sales`:** one row per order line item, joined to `orders` (for `Customer_ID`), `products` (for pricing, filtered to `is_current = true`), `dim_date` (for `Time_ID`), `address` (primary shipping address per customer), and `payments` (1:1 on `Order_ID`). The one surrogate key in the whole table is `fact_sales_sk = sha2(order_item_id)` — everything else is a natural key. Day 7's version rebuilds this from scratch every run (`mode("overwrite")`). **This HOL keeps the exact same join logic, but scopes it to only the order_items that changed, and writes with `MERGE` instead of `overwrite`.**

**Instructions:** Replace `YOUR_SCHEMA` below with the same one you used in HOL 1. Run cells in order.

## Setup — Clone the Sources + the Target

In [ ]:
# ─── Use the same schema you created in HOL 1 ───
YOUR_SCHEMA = "main.YOUR_SCHEMA"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {YOUR_SCHEMA}")

SOURCE_TABLES = ["order_items", "orders", "products", "address", "payments"]
for t in SOURCE_TABLES:
    spark.sql(f"CREATE OR REPLACE TABLE {YOUR_SCHEMA}.silver_{t} SHALLOW CLONE gbmart.silver.{t}")

spark.sql(f"CREATE OR REPLACE TABLE {YOUR_SCHEMA}.dim_date   SHALLOW CLONE gbmart.gold.dim_date")
spark.sql(f"CREATE OR REPLACE TABLE {YOUR_SCHEMA}.fact_sales SHALLOW CLONE gbmart.gold.fact_sales")

# CDF is what lets us find only the CHANGED order_items — enable it on the clone
spark.sql(f"ALTER TABLE {YOUR_SCHEMA}.silver_order_items SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"Cloned into {YOUR_SCHEMA}: silver_{{order_items,orders,products,address,payments}}, dim_date, fact_sales")

## Step 1 — Find the Starting Point for CDF

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

ORDER_ITEMS_TABLE = f"{YOUR_SCHEMA}.silver_order_items"

LAST_PROCESSED_VERSION = spark.sql(f"DESCRIBE HISTORY {ORDER_ITEMS_TABLE}").selectExpr("max(version)").collect()[0][0]
print(f"LAST_PROCESSED_VERSION = {LAST_PROCESSED_VERSION}")
print(f"fact_sales currently has {spark.table(f'{YOUR_SCHEMA}.fact_sales').count():,} rows")

## Simulate an Incoming Change

In production, new `order_items` rows arrive via the normal Bronze→Silver pipeline whenever a customer places an order. For this hands-on, insert a couple of new order-line rows directly into your Silver clone, so CDF has a real, fresh commit to detect.

In [ ]:
# YOUR CODE HERE — build 2 new order_items rows for an EXISTING order_id and product_id,
# so the joins below (to orders/products/etc.) resolve to real dimension data.
# Hint: pick a real order_id from silver_orders and a real product_id from silver_products.

existing_order_id = spark.table(f"{YOUR_SCHEMA}.silver_orders").select("order_id").limit(1).collect()[0][0]
existing_product_id = spark.table(f"{YOUR_SCHEMA}.silver_products") \
    .filter("is_current = true").select("product_id").limit(1).collect()[0][0]

new_order_items_df = spark.createDataFrame([
    ("DEMO-OI-9001", existing_order_id, existing_product_id, 2),
    ("DEMO-OI-9002", existing_order_id, existing_product_id, 1),
], ["order_item_id", "order_id", "product_id", "quantity"])

new_order_items_df.write.format("delta").mode("append").saveAsTable(ORDER_ITEMS_TABLE)
print(f"Inserted 2 new order_items rows against order_id={existing_order_id}, product_id={existing_product_id}")

## Step 2 — Identify Only the Changed `order_items` (CDF)

This is the entire point of doing this incrementally: instead of re-joining all ~377K `fact_sales` rows worth of order_items, we isolate just the handful that actually changed.

In [ ]:
changed_order_items_df = (
    spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", LAST_PROCESSED_VERSION + 1)
        .table(ORDER_ITEMS_TABLE)
        .filter("_change_type != 'update_preimage'")
        .select("order_item_id", "order_id", "product_id", "quantity")
)

print(f"Changed order_items rows: {changed_order_items_df.count():,}")
changed_order_items_df.display()

## Step 3 — Rebuild the Day 7 Join Chain, Scoped to Just These Rows

Identical logic to Day 7's `fact_sales` build — `order_items` → `orders` (for `Customer_ID`) → `products` (current price) → `dim_date` (for `Time_ID`) → `address` (primary shipping address) → `payments` (1:1 on order) — just starting from `changed_order_items_df` instead of the full table.

In [ ]:
# Same column renames Day 7 used, applied to just the changed rows
base_df = changed_order_items_df.select(
    col("order_item_id"),
    col("order_id").alias("Order_ID"),
    col("product_id").alias("Product_ID"),
    col("quantity").alias("Quantity_purchased")
)

orders_df = spark.table(f"{YOUR_SCHEMA}.silver_orders") \
    .select(col("order_id").alias("Order_ID"), col("customer_id").alias("Customer_ID"), "order_date")

base_df = base_df.join(orders_df, "Order_ID")
print(f"Base rows (changed order_items joined to orders): {base_df.count():,}")

In [ ]:
# Products — filter to is_current=true so each product resolves to exactly one price row
products_current = spark.table(f"{YOUR_SCHEMA}.silver_products").filter("is_current = true") \
    .select(
        col("product_id").alias("Product_ID"),
        col("actual_price_inr").alias("Actual_price"),
        col("discounted_price_inr").alias("Discounted_price")
    )

enriched_df = base_df.join(products_current, "Product_ID")
print(f"Rows after products join: {enriched_df.count():,}")

In [ ]:
# dim_date — resolve Time_ID from order_date
dim_date_df = spark.table(f"{YOUR_SCHEMA}.dim_date").select("date", col("date_key").alias("Time_ID"))

enriched_df = enriched_df.join(dim_date_df, enriched_df.order_date == dim_date_df.date, "left").drop("date")
print(f"Rows after dim_date join: {enriched_df.count():,}")

In [ ]:
# address — a customer can have more than one address; prefer one whose
# address_type includes "Shipping", otherwise take the first available (same rule as Day 7)
address_window = Window.partitionBy("customer_id").orderBy(
    when(col("address_type").contains("Shipping"), 0).otherwise(1)
)

address_primary = spark.table(f"{YOUR_SCHEMA}.silver_address") \
    .withColumn("_rank", row_number().over(address_window)) \
    .filter("_rank = 1") \
    .select(col("customer_id").alias("Customer_ID"), col("address_id").alias("Address_ID"))

enriched_df = enriched_df.join(address_primary, "Customer_ID", "left")

# payments — 1:1 with orders
payments_df = spark.table(f"{YOUR_SCHEMA}.silver_payments") \
    .select(col("order_id").alias("Order_ID"), col("payment_id").alias("Payment_ID"))

enriched_df = enriched_df.join(payments_df, "Order_ID", "left")
print(f"Rows after address + payments joins: {enriched_df.count():,}")

## Step 4 — Compute the Fact Row(s)

Same `fact_sales_sk` and `Sales_amount` formula as Day 7 — nothing changes about *how* a fact row is shaped, only *how many* rows we're computing it for.

In [ ]:
new_fact_rows_df = enriched_df \
    .withColumn("fact_sales_sk", sha2(col("order_item_id"), 256)) \
    .withColumn("Sales_amount", col("Quantity_purchased") * col("Discounted_price")) \
    .select(
        "fact_sales_sk", "Payment_ID", "Customer_ID", "Product_ID", "Order_ID", "Address_ID",
        "Time_ID", "Quantity_purchased", "Actual_price", "Discounted_price", "Sales_amount"
    )

new_fact_rows_df.display()

## Step 5 — `MERGE INTO fact_sales` (Not `overwrite`)

This is the one line that's genuinely different from Day 7. Keyed on `fact_sales_sk` — since that's a deterministic hash of `order_item_id`, a row that already exists will always match itself on re-run, so this MERGE is safe to run repeatedly without creating duplicates.

- **`WHEN MATCHED`** — an existing order_item's fact row got recomputed (e.g. a late-arriving price correction). Update it.
- **`WHEN NOT MATCHED`** — a genuinely new order_item. Insert it.

In [ ]:
fact_sales_table = DeltaTable.forName(spark, f"{YOUR_SCHEMA}.fact_sales")

(fact_sales_table.alias("tgt")
    .merge(new_fact_rows_df.alias("src"), "tgt.fact_sales_sk = src.fact_sales_sk")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE complete")

## Step 6 — Verify

In [ ]:
fact_df = spark.table(f"{YOUR_SCHEMA}.fact_sales")
print(f"fact_sales rows now: {fact_df.count():,}")

fact_df.filter(col("fact_sales_sk").isin(
    [row.fact_sales_sk for row in new_fact_rows_df.select("fact_sales_sk").collect()]
)).display()

**Q: Your new order_items rows should now appear in `fact_sales` with a correctly computed `Sales_amount`. Confirm the row count grew by exactly the number of new order_items you inserted.**

*Your answer:* _______________

## Recap — Full Overwrite vs. Incremental MERGE

| | Day 7's approach (`overwrite`) | This HOL's approach (`MERGE`) |
|---|---|---|
| **Rows reprocessed per run** | All ~377K | Only what changed since last run |
| **Cost at GlobalMart's current scale** | Acceptable — a training-scale rebuild is cheap | Marginal — but the win compounds as the table grows |
| **Complexity** | Simpler — one join, one write | Requires tracking `LAST_PROCESSED_VERSION` and CDF |
| **When to choose which** | Small-to-medium fact tables, or when a full rebuild is simpler to reason about | Large fact tables where a full rescan becomes the bottleneck |

Both are valid engineering choices — the point isn't that MERGE is always "better," it's that you now have both tools and can pick based on table size and how expensive a full rebuild actually is.